In [3]:
import duckdb

# Connect to an in-memory DuckDB instance
con = duckdb.connect()

# Define the folder where your individual sensor files are saved
input_folder = "floodnet_parquet_data/*.parquet"

print("Merging all sensors into a single massive Parquet file...")
# We use read_parquet with union_by_name=True to prevent schema mismatch errors
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{input_folder}', union_by_name=True)
    ) 
    TO 'Data_Files/floodnet_full_dataset_merged.parquet' (FORMAT PARQUET);
""")
print("Full dataset merged successfully!")

print("Extracting flood events (>10mm processed depth) into a separate file...")
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{input_folder}', union_by_name=True)
        WHERE depth_proc_mm > 10
    ) 
    TO 'Data_Files/floodnet_floods_only.parquet' (FORMAT PARQUET);
""")
print("Flood events isolated and saved successfully!")

# Optional: Print out the final row counts to verify
total_rows = con.execute("SELECT count(*) FROM 'Data_Files/floodnet_full_dataset_merged.parquet'").fetchone()[0]
flood_rows = con.execute("SELECT count(*) FROM 'Data_Files/floodnet_floods_only.parquet'").fetchone()[0]

print(f"\nSummary:")
print(f"Total rows across all sensors: {total_rows:,}")
print(f"Total rows with flood events:  {flood_rows:,}")

Merging all sensors into a single massive Parquet file...
Full dataset merged successfully!
Extracting flood events (>10mm processed depth) into a separate file...
Flood events isolated and saved successfully!

Summary:
Total rows across all sensors: 164,302,033
Total rows with flood events:  8,452,069
